In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential, Model
from keras.layers import LSTM, Dense, Dropout, Input, RepeatVector
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report, matthews_corrcoef
import matplotlib.pyplot as plt
import scikitplot as skplt
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM, GRU
from keras.layers import Dense , Dropout
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import scikitplot as skplt
from sklearn.preprocessing import label_binarize
from sklearn.datasets import load_digits
#from yellowbrick.features import ParallelCoordinates
from keras.layers import Bidirectional, LSTM
from keras.layers import Bidirectional, GRU
from sklearn.metrics import matthews_corrcoef

In [ ]:
filepath= "C:/Users/Theda/Desktop/Edge-iiotset.csv"

In [ ]:
mydata= pd.read_csv(filepath)

In [ ]:
mydata.info()

In [ ]:
drop_columns = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4","arp.dst.proto_ipv4", 

         "http.file_data","http.request.full_uri","icmp.transmit_timestamp",

         "http.request.uri.query", "tcp.options","tcp.payload","tcp.srcport",

         "tcp.dstport", "udp.port", "mqtt.msg"]

mydata.drop(drop_columns, axis=1, inplace=True)


In [ ]:
print(mydata['Attack_type'].value_counts())

In [ ]:
mydata.drop('Attack_label', axis='columns', inplace=True)


In [ ]:
mydata.replace([np.inf, -np.inf], np.nan, inplace=True)
mydata.dropna(inplace=True)
mydata.drop_duplicates(inplace=True)
mydata.reset_index(inplace=True, drop=True)
mydata = mydata.sample(frac=1).reset_index(drop=True)

In [ ]:
mydata.info()

In [ ]:
attacks = {'Normal': 0 ,'DDoS_UDP' :1, 'DDoS_ICMP':2, 'SQL_injection':3, 'Password':4,
       'Vulnerability_scanner':5 , 'DDoS_TCP':6, 'DDoS_HTTP':7, 'Uploading':8, 'Backdoor':9, 
       'Port_Scanning':10, 'XSS':11, 'Ransomware':12, 'Fingerprinting':13, 'MITM':14}
mydata['label']=mydata['Attack_type'].map(attacks)

In [ ]:
print(mydata['label'].value_counts())

In [ ]:
mydata.drop('Attack_type', axis='columns', inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()

In [ ]:
mydata['mqtt.protoname'] = mydata['mqtt.protoname'].astype(str)
mydata['mqtt.topic'] = mydata['mqtt.topic'].astype(str)
mydata['mqtt.conack.flags'] = mydata['mqtt.conack.flags'].astype(str)
mydata['dns.qry.name.len'] = mydata['dns.qry.name.len'].astype(str)
mydata['http.request.method'] = mydata['http.request.method'].astype(str)
mydata['http.referer'] = mydata['http.referer'].astype(str)
mydata['http.request.version'] = mydata['http.request.version'].astype(str)

In [ ]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()

mydata['mqtt.protoname'] = labelencoder.fit_transform(mydata['mqtt.protoname'])
mydata['mqtt.topic'] = labelencoder.fit_transform(mydata['mqtt.topic'])
mydata['mqtt.conack.flags'] = labelencoder.fit_transform(mydata['mqtt.conack.flags'])
mydata['dns.qry.name.len'] = labelencoder.fit_transform(mydata['dns.qry.name.len'])
mydata['http.request.method'] = labelencoder.fit_transform(mydata['http.request.method'])
mydata['http.referer'] = labelencoder.fit_transform(mydata['http.referer'])
mydata['http.request.version'] = labelencoder.fit_transform(mydata['http.request.version'])


In [ ]:
mydata.info()

In [ ]:
X = mydata.iloc[:, 0:46]
Y = mydata.iloc[:, 46:47]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler()
X = sc.fit_transform(X)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.30, random_state=42)
x_train = np.array(x_train)
x_train = np.reshape (x_train,(x_train.shape[0], 1, x_train.shape[1]))
#Reshape and normalize test data
x_test = np.array(x_test)
x_test = np.reshape (x_test,(x_test.shape[0], 1, x_test.shape[1]))

In [ ]:
Num_classes = len(np.unique(Y))
y_train_ohe=to_categorical(y_train,Num_classes)
y_train_ohe = pd.DataFrame(y_train_ohe)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import numpy as np
import tensorflow as tf

In [ ]:
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    dim = K.int_shape(z_mean)[1]
    epsilon = K.random_normal(shape=(batch, dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

In [ ]:
beta = 4.0  # (beta >= 1)

In [ ]:
input_dim = x_train.shape[2]  # This should be 8, representing data[0] to data[7]
inputs = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(inputs)
z_mean = Dense(32)(encoded)
z_log_var = Dense(32)(encoded)

In [ ]:
z = Lambda(sampling, output_shape=(32,))([z_mean, z_log_var])


In [ ]:
decoded = Dense(64, activation='relu')(z)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

In [ ]:
vae = Model(inputs, decoded)

In [ ]:
class KLLossLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(KLLossLayer, self).__init__(**kwargs)
    
    def call(self, inputs):
        z_mean, z_log_var = inputs
        kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
        self.add_loss(beta * K.mean(kl_loss))  # Add the KL loss to the final model loss
        return z_mean

In [ ]:
z_mean_kl_loss = KLLossLayer()([z_mean, z_log_var])

In [ ]:
def beta_vae_loss(inputs, decoded):
    # Reconstruction loss (Mean Squared Error)
    reconstruction_loss = K.mean(K.square(inputs - decoded), axis=-1)
    return reconstruction_loss  # KL loss is added automatically

In [ ]:
vae.compile(optimizer='adam', loss=beta_vae_loss)


In [ ]:
# Reshape x_train to match the expected input shape (batch_size, input_dim)
x_train_flattened = x_train.reshape((x_train.shape[0], x_train.shape[2]))

In [ ]:
# Train the Beta-VAE
betahistory=vae.fit(x_train_flattened, x_train_flattened, epochs=20, batch_size=1024, validation_split=0.5)

In [ ]:
encoder = Model(inputs, z_mean)  # Use the mean as the encoded representation
x_train_encoded = encoder.predict(x_train_flattened)
x_test_encoded = encoder.predict(x_test.reshape((x_test.shape[0], x_test.shape[2])))

x_train_encoded = np.reshape(x_train_encoded, (x_train_encoded.shape[0], 1, x_train_encoded.shape[1]))
x_test_encoded = np.reshape(x_test_encoded, (x_test_encoded.shape[0], 1, x_test_encoded.shape[1]))

In [ ]:
#Fewer attention heads and scales with residual Block

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Attention, Input, Dense, LSTM, Bidirectional, Conv1D, LayerNormalization, Add, Dropout, Concatenate, Flatten
from tensorflow.keras.models import Model
# Enhanced Multi-Scale Multi-Head Attention Layer
class MultiScaleMultiHeadAttention(Layer):
    def __init__(self, num_heads=1, scales=[1], feature_dim=None, **kwargs):
        super(MultiScaleMultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.scales = scales
        self.conv_layers = [
            [Conv1D(filters=feature_dim, kernel_size=scale, padding='same', activation='relu')
             for _ in range(num_heads)] for scale in scales]
        self.attention_heads = [[Attention() for _ in range(num_heads)] for _ in self.scales]
        self.fusion_layer = Dense(feature_dim * len(scales) * num_heads, activation='sigmoid')

    def call(self, x):
        multi_scale_outputs = []
        for scale_idx, (scale_heads, scale_conv_layers) in enumerate(zip(self.attention_heads, self.conv_layers)):
            for head_idx, (attention, conv) in enumerate(zip(scale_heads, scale_conv_layers)):
                conv_x = conv(x)
                attention_output = attention([conv_x, conv_x])
                multi_scale_outputs.append(attention_output)
        
        concatenated = Concatenate(axis=-1)(multi_scale_outputs)
        fusion_weight = self.fusion_layer(concatenated)
        fused_output = concatenated * fusion_weight
        return fused_output
        # Residual BiLSTM block with matching dimensions
def residual_bilstm_block(x, units):
    x_res = x  # Save residual connection
    x = Bidirectional(LSTM(units, activation='relu', return_sequences=True))(x)
    x = LayerNormalization()(x)
    
    # Match dimensions of residual connection using a Dense layer
    x_res = Dense(units * 2)(x_res)  # Since Bidirectional doubles the units
    x = Add()([x, x_res])  # Adding the residual connection
    return x
# Assuming x_train_encoded.shape[2] gives the number of features per timestep
num_features = x_train_encoded.shape[2]  # You should define x_train_encoded.shape[2] before this script
# Define the enhanced model with Multi-Scale Temporal Attention and Adaptive Feature Fusion
inputs = Input(shape=(x_train_encoded.shape[1], x_train_encoded.shape[2]))
x = Bidirectional(LSTM(80, activation='relu', return_sequences=True))(inputs)
# Adding residual BiLSTM blocks with dimension matching
x = residual_bilstm_block(x, 32)
x = residual_bilstm_block(x, 16)
x = Dropout(0.1)(x)
# Multi-Scale Multi-Head Attention
x = MultiScaleMultiHeadAttention(num_heads=1, scales=[1], feature_dim=num_features)(x)
x = Flatten()(x)  # Flatten to ensure the output dimension is correct
x = Dropout(rate=0.1)(x)
outputs = Dense(15, activation='softmax')(x)  # Update Num_classes to your actual number of classes

model = Model(inputs=inputs, outputs=outputs)
# Compile with AdamW and custom learning rate schedule
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4), metrics=['accuracy'])

# Custom learning rate schedule
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)

In [ ]:
history1 = model.fit(x_train_encoded, y_train_ohe, epochs=20, validation_split=0.5, batch_size=1024, callbacks=[lr_schedule])

In [ ]:
predict_prob1=model.predict([x_test_encoded])
predict_classes=np.argmax(predict_prob1,axis=1)
prediction1 = model.predict(x_test_encoded)
prediction1 =prediction1.argmax(1)

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction1)* 100)) 
print(precision_score(y_test, prediction1, average="macro"))  
print(recall_score(y_test, prediction1, average="macro"))
print(f1_score(y_test, prediction1, average="macro"))

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction1)* 100)) 
print(precision_score(y_test, prediction1, average="weighted"))  
print(recall_score(y_test, prediction1, average="weighted"))
print(f1_score(y_test, prediction1, average="weighted"))

In [ ]:
#Fewer attention heads and scales and No resiudal Block

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Attention, Input, Dense, LSTM, Bidirectional, Conv1D, LayerNormalization, Add, Dropout, Concatenate, Flatten
from tensorflow.keras.models import Model
# Enhanced Multi-Scale Multi-Head Attention Layer
class MultiScaleMultiHeadAttention(Layer):
    def __init__(self, num_heads=1, scales=[1], feature_dim=None, **kwargs):
        super(MultiScaleMultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.scales = scales
        self.conv_layers = [
            [Conv1D(filters=feature_dim, kernel_size=scale, padding='same', activation='relu')
             for _ in range(num_heads)] for scale in scales]
        self.attention_heads = [[Attention() for _ in range(num_heads)] for _ in self.scales]
        self.fusion_layer = Dense(feature_dim * len(scales) * num_heads, activation='sigmoid')

    def call(self, x):
        multi_scale_outputs = []
        for scale_idx, (scale_heads, scale_conv_layers) in enumerate(zip(self.attention_heads, self.conv_layers)):
            for head_idx, (attention, conv) in enumerate(zip(scale_heads, scale_conv_layers)):
                conv_x = conv(x)
                attention_output = attention([conv_x, conv_x])
                multi_scale_outputs.append(attention_output)
        
        concatenated = Concatenate(axis=-1)(multi_scale_outputs)
        fusion_weight = self.fusion_layer(concatenated)
        fused_output = concatenated * fusion_weight
        return fused_output
# Assuming x_train_encoded.shape[2] gives the number of features per timestep
num_features = x_train_encoded.shape[2]  # You should define x_train_encoded.shape[2] before this script
# Define the enhanced model with Multi-Scale Temporal Attention and Adaptive Feature Fusion
inputs = Input(shape=(x_train_encoded.shape[1], x_train_encoded.shape[2]))
x = Bidirectional(LSTM(80, activation='relu', return_sequences=True))(inputs)
x = Dropout(0.1)(x)
# Multi-Scale Multi-Head Attention
x = MultiScaleMultiHeadAttention(num_heads=1, scales=[1], feature_dim=num_features)(x)
x = Flatten()(x)  # Flatten to ensure the output dimension is correct
x = Dropout(rate=0.1)(x)
outputs = Dense(15, activation='softmax')(x)  # Update Num_classes to your actual number of classes

model = Model(inputs=inputs, outputs=outputs)
# Compile with AdamW and custom learning rate schedule
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4), metrics=['accuracy'])

# Custom learning rate schedule
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)

In [ ]:
history2 = model.fit(x_train_encoded, y_train_ohe, epochs=20, validation_split=0.5, batch_size=1024, callbacks=[lr_schedule])

In [ ]:
predict_prob2=model.predict([x_test_encoded])
predict_classes=np.argmax(predict_prob2,axis=1)
prediction2 = model.predict(x_test_encoded)
prediction2 =prediction2.argmax(1)

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction2)* 100)) 
print(precision_score(y_test, prediction2, average="macro"))  
print(recall_score(y_test, prediction2, average="macro"))
print(f1_score(y_test, prediction2, average="macro"))

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction2)* 100)) 
print(precision_score(y_test, prediction2, average="weighted"))  
print(recall_score(y_test, prediction2, average="weighted"))
print(f1_score(y_test, prediction2, average="weighted"))

In [ ]:
#Without residuls and MSTAFF-AFF Function

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Attention, Input, Dense, LSTM, Bidirectional, Conv1D, LayerNormalization, Add, Dropout, Concatenate, Flatten
from tensorflow.keras.models import Model
num_features = x_train_encoded.shape[2]  # You should define x_train_encoded.shape[2] before this script
# Define the enhanced model with Multi-Scale Temporal Attention and Adaptive Feature Fusion
inputs = Input(shape=(x_train_encoded.shape[1], x_train_encoded.shape[2]))
x = Bidirectional(LSTM(80, activation='relu', return_sequences=True))(inputs)
x = Dropout(0.1)(x)
x = Flatten()(x)  # Flatten to ensure the output dimension is correct
x = Dropout(rate=0.1)(x)
outputs = Dense(15, activation='softmax')(x)  # Update Num_classes to your actual number of classes

model = Model(inputs=inputs, outputs=outputs)
# Compile with AdamW and custom learning rate schedule
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4), metrics=['accuracy'])

# Custom learning rate schedule
lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

In [ ]:
history3 = model.fit(x_train_encoded, y_train_ohe, epochs=20, validation_split=0.5, batch_size=1024, callbacks=[lr_schedule])

In [ ]:
predict_prob3=model.predict([x_test_encoded])
predict_classes=np.argmax(predict_prob3,axis=1)
prediction3 = model.predict(x_test_encoded)
prediction3 =prediction3.argmax(1)

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction3)* 100)) 
print(precision_score(y_test, prediction3, average="macro"))  
print(recall_score(y_test, prediction3, average="macro"))
print(f1_score(y_test, prediction3, average="macro"))

In [ ]:
print("Accuracy:" + str(accuracy_score(y_test, prediction3)* 100)) 
print(precision_score(y_test, prediction3, average="weighted"))  
print(recall_score(y_test, prediction3, average="weighted"))
print(f1_score(y_test, prediction3, average="weighted"))